In [1]:
import pandas as pd
import numpy as np
import glob
import os
import sys
sys.path.insert(0, '../')
#sys.path.insert(0, './')
from data.Reinhard import Reinhard
from models.model_mrcnn import _default_mrcnn_config, build_default
from visualization.explain import ExplainPredictions
import torch
import tqdm
from PIL import Image
import torchvision
from torchmetrics.classification import MulticlassConfusionMatrix
import torchvision.ops.boxes as bops
from torchmetrics.classification import Dice
from PIL import Image
from skimage import draw
from skimage import measure
import plotly.express as px

In [2]:
dice = Dice(average='samples')
preds = torch.tensor([2, 0, 2, 1])
target = torch.tensor([1, 1, 2, 0])
dice(preds, target)

tensor(0.2500)

In [3]:
LBD_model_path = '/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/models/mrcnn_models/woven-wood-253_mrcnn_model_24.pth'

In [4]:
test_config = dict(batch_size = 1, num_classes = 2)
model_config = _default_mrcnn_config(num_classes=1 + test_config['num_classes']).config
model_lbd = build_default(model_config, im_size=1024)

/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/torchvision/models/_utils.py:135: UserWarning: Using 'backbone_name' as positional parameter(s) is deprecated since 0.13 and may be removed in the future. Please use keyword parameter(s) instead.
  warnings.warn(
/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/mahirwar/miniconda3/envs/kfold_amy_plaque1/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet152_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet152_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [5]:
model_lbd.load_state_dict(torch.load(LBD_model_path))

<All keys matched successfully>

In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_lbd.eval().to(device)

GeneralizedRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.8198, 0.7572, 0.7618], std=[0.0848, 0.1066, 0.1211])
      Resize(min_size=(1024,), max_size=1024, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=1e-05)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=1e-05)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=1e-05)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=1e-05)
   

In [11]:
def prepare_input(image):
    image_float_np = np.float32(image) / 255

    # define the torchvision image transforms
    transform = torchvision.transforms.Compose([
        torchvision.transforms.ToTensor(),
    ])
    input_tensor = transform(image)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    input_tensor = input_tensor.to(device)
    # Add a batch dimension:
    input_tensor = input_tensor.unsqueeze(0)
    return input_tensor, image_float_np

class_names = ['True', 'Pre']
def get_outputs(input_tensor, model, threshold):
    with torch.no_grad():
        # forward pass of the image through the modle
        outputs = model(input_tensor)
    #print(outputs)
    scores = list(outputs[0]['scores'].detach().cpu().numpy())
    # print("\n scores", max(scores))
    # index of those scores which are above a certain threshold
    thresholded_preds_inidices = [scores.index(i) for i in scores if i > threshold]
    #print(thresholded_preds_inidices)
    thresholded_preds_count = len(thresholded_preds_inidices)
    #print(thresholded_preds_count)
    scores = scores[:thresholded_preds_count]
    # get the masks
    masks = (outputs[0]['masks']>0.5).squeeze().detach().cpu().numpy()
    # print("masks", masks)
    # discard masks for objects which are below threshold
    masks = masks[:thresholded_preds_count]
    # get the bounding boxes, in (x1, y1), (x2, y2) format
    boxes = [[(int(i[0]), int(i[1])), (int(i[2]), int(i[3]))]  for i in outputs[0]['boxes'].detach().cpu()]
    # discard bounding boxes below threshold value
    boxes = boxes[:thresholded_preds_count]
    # get the classes labels
    # print('labels', outputs[0]['labels'])
    #print(outputs[0]['labels'])
    #print(thresholded_preds_count)
    #print(outputs[0]['labels'])
    labels = [class_names[i-1] for i in outputs[0]['labels']]
    #labels = [i for i in outputs[0]['labels']]
    #print(labels)
    labels = labels[:thresholded_preds_count]
    return masks, boxes, labels, scores

In [12]:
def expand_mask(mask_path):
    mask = Image.open(mask_path).convert('P')   
    mask = np.array(mask)
    # instances are encoded as different colors
    obj_ids = np.unique(mask)
    # first id is the background, so remove it
    obj_ids = obj_ids[1:]
    # split the color-encoded mask into a set
    # of binary masks
    masks = mask == obj_ids[:, None, None]
    num_objs = len(obj_ids)
    boxes = []
    for i in range(num_objs):
        pos = np.where(masks[i])
        xmin = np.min(pos[1])
        xmax = np.max(pos[1])
        ymin = np.min(pos[0])
        ymax = np.max(pos[0])
        #print(pos)
        if xmax <= xmin and ymax <=ymin:
            print("degenrate boxes", mask_path)
            print(len(obj_ids))
            break
        boxes.append([xmin, ymin, xmax, ymax])
    boxes = torch.as_tensor(boxes, dtype=torch.float32)
    x = [id // 50 for id in obj_ids]
    labels = torch.tensor(x)
    masks = torch.as_tensor(masks, dtype=torch.uint8)
    #image_id = torch.tensor([idx])
    area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
    # suppose all instances are not crowd
    iscrowd = torch.zeros((num_objs,), dtype=torch.int64)
    target = {}
    target["boxes"] = boxes
    target["labels"] = labels
    target["masks"] = masks
    #target["image_id"] = image_id
    target["area"] = area
    target["iscrowd"] = iscrowd
    return target

In [13]:
def match_mask(device,masked_image,binary_array):
    #num_classes = len(set(list(np.unique(masked_image)) +  list(np.unique(binary_array))))
    num_classes = 2
    metric = MulticlassConfusionMatrix(num_classes=num_classes).to(device)
    conf_final=torch.tensor(np.zeros((num_classes,num_classes))).to(device).to(torch.int64)
    for i in range(len(masked_image)):
        preds = torch.tensor(masked_image[i]).to(device).to(torch.int64)
        target = torch.tensor(binary_array[i]).to(device).to(torch.int64)
        a1 = metric(preds,target)
        conf_final = conf_final + a1     
    conf_final_np = conf_final.cpu().numpy()
    total_predicted = np.sum(conf_final_np, axis=0) 
    diag_elements = np.diag(conf_final_np)
    precision = diag_elements/total_predicted
    total_actual = np.sum(conf_final_np, axis=1) 
    recall = diag_elements/total_actual
    f1_score = (2*precision*recall)/(recall+precision)
    iou_coeff = (diag_elements)/(total_predicted+total_actual-diag_elements)
    
    #csv_filename_tosave = "Eval_Metric_"+ geofile.split(".")[0] + ".csv"
    eval_metrics = pd.DataFrame({"Class":["Background","Pre/True"],"Precision":precision, "recall":recall,"f1_score":f1_score,"iou_coeff":iou_coeff})
    #eval_metrics.to_csv(os.path.join(eval_dir,csv_filename_tosave))
    return eval_metrics[eval_metrics["Class"]=="Pre/True"]["f1_score"].values[0]

In [14]:
def match_label(pred_label, gt_label):
    if pred_label==gt_label:
        return True
    else:
        return False

def actual_label_target(gt_label):
    if (gt_label.cpu().numpy()==1):
        return "True"
    if (gt_label.cpu().numpy()==2):
        return "Pre"
    return None

In [15]:
test_folder  = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth"
image_folders = glob.glob(os.path.join(test_folder, '*.svs'))
output_df =  pd.DataFrame()
for image_folder in image_folders:
    images = glob.glob(os.path.join(image_folder, "images", "*.png"))
    masks = glob.glob(os.path.join(image_folder, "labels", "*.png"))
    images.sort()
    masks.sort()
    img_list = []
    target_mask_list= []
    pred_mask_list=[]
    f1_score_list = []
    matched_label_list = []
    actual_label_list = []
    pred_label_list =[]
    for img, mask in zip(images,masks):
        #img_name = os.path.basename(img).split('.')[0]
        image = np.array(Image.open(img))
        #mask = np.array(Image.open(mask))
        target = expand_mask(mask)
        if image.shape[2] == 4:
            image = image[:,:, :3]
        input_tensor, image_float_np = prepare_input(image)
        masks, boxes, labels, scores = get_outputs(input_tensor, model_lbd, 0.65)
        for i in range(len(target['masks'])):
            target_label = actual_label_target(target['labels'][i])
            for j in range(len(masks)):
                if len(masks[j].shape)>=2:
                    actual_label_list.append(target_label)
                    pred_label_list.append(labels[j])
                    f1_score = match_mask(device, masks[j],target['masks'][i])
                    print(f1_score)
                    matched_label = match_label(labels[j],target_label)
                    print(matched_label)
                    img_list.append(img)
                    target_mask_list.append(i)
                    pred_mask_list.append(j)
                    f1_score_list.append(f1_score)
                    matched_label_list.append(matched_label)
                    
    tmp = pd.DataFrame({"img_crop":img_list, "target_label":actual_label_list, "pred_label":pred_label_list, "target_mask_index":target_mask_list,"pred_mask_index":pred_mask_list, "dice/f1_score":f1_score_list,"matched_label":matched_label_list})
    tmp["image_folder"] = image_folder
    if len(output_df)==0:
        output_df = tmp
    else:
        output_df= pd.concat([output_df,tmp],ignore_index=True)


/tmp/ipykernel_750504/3669700039.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(binary_array[i]).to(device).to(torch.int64)


0.8641025641025641
True
0.7939508506616257
True
0.9466135458167331
True
0.9378407851690295
True
0.9001536098310293
True


/tmp/ipykernel_750504/3669700039.py:17: RuntimeWarning: invalid value encountered in true_divide
  f1_score = (2*precision*recall)/(recall+precision)


nan
True
0.9106145251396648
True
0.8760611205432937
True
0.8583569405099151
True
0.9086357947434293
True
0.9095182138660399
False
nan
True
0.8844086021505377
True
0.8267223382045927
True
0.8725314183123878
True
nan
False
0.929305912596401
True
nan
False
0.9219701162147206
True
0.8786264061574897
True
0.8746081504702194
True
nan
True
0.9325153374233129
True


/tmp/ipykernel_750504/3669700039.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(binary_array[i]).to(device).to(torch.int64)


...

nan
True
0.9232954545454545
True
0.9555140186915887
True
nan
False
0.9450207468879668
True
nan
False
0.6666666666666667
False
0.9096844396082698
False
nan
False
nan
False
0.9672991522002422
True
0.9384057971014492
True
0.931958762886598
True
nan
True
0.8448145344436032
False
0.9028177113283496
False
0.8976377952755906
False
0.9445968184311574
True
0.9295967190704032
True
0.9211409395973155
True
0.9371513573819263
True
0.8980755523877405
False
nan
False
0.8615149196633511
False
0.9485776805251641
True
0.8283038501560874
True
0.9189660180075515
False


/tmp/ipykernel_750504/3669700039.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(binary_array[i]).to(device).to(torch.int64)
/tmp/ipykernel_750504/3669700039.py:17: RuntimeWarning: invalid value encountered in true_divide
  f1_score = (2*precision*recall)/(recall+precision)


nan
False
0.8568486096807416
True
0.8099547511312218
False
0.9334836527621195
True
0.8970331588132636
True
nan
False
0.9011913104414856
True
nan
False
0.9026845637583892
True
0.7695906432748537
True
0.9416846652267817
True
0.8369477911646587
False


/tmp/ipykernel_750504/1671116192.py:46: FutureWarning: Behavior when concatenating bool-dtype and numeric-dtype arrays is deprecated; in a future version these will cast to object dtype (instead of coercing bools to numeric values). To retain the old behavior, explicitly cast bool-dtype arrays to numeric dtype.
  output_df= pd.concat([output_df,tmp],ignore_index=True)
/tmp/ipykernel_750504/3669700039.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  target = torch.tensor(binary_array[i]).to(device).to(torch.int64)
/tmp/ipykernel_750504/3669700039.py:17: RuntimeWarning: invalid value encountered in true_divide
  f1_score = (2*precision*recall)/(recall+precision)


nan
False
0.8600508905852418
False
0.907185628742515
False
0.772823779193206
True
0.8676671214188267
True
0.7043010752688172
True
nan
False
0.7999999999999999
False
0.8
False
0.7719298245614036
True
0.8985801217038539
True
0.8272727272727273
False
0.9067164179104479
False
nan
False
0.7972972972972973
False
nan
True
0.900900900900901
True


In [16]:
output_df["image_name"] =output_df["img_crop"].apply(lambda l:l.split("/")[-1])

In [17]:
output_df.head()

,img_crop,target_label,pred_label,target_mask_index,pred_mask_index,dice/f1_score,matched_label,image_folder,image_name
0,/gladstone/finkbeiner/steve/work/data/npsad_da...,True,True,0.0,0.0,0.864103,1.0,/gladstone/finkbeiner/steve/work/data/npsad_da...,PD110_Syn1_TCx_10240x_34816y_image.png
1,/gladstone/finkbeiner/steve/work/data/npsad_da...,Pre,Pre,0.0,0.0,0.793951,1.0,/gladstone/finkbeiner/steve/work/data/npsad_da...,PD110_Syn1_TCx_18432x_15360y_image.png
2,/gladstone/finkbeiner/steve/work/data/npsad_da...,True,True,0.0,0.0,0.946614,1.0,/gladstone/finkbeiner/steve/work/data/npsad_da...,PD110_Syn1_TCx_32768x_4096y_image.png
3,/gladstone/finkbeiner/steve/work/data/npsad_da...,True,True,0.0,0.0,0.937841,1.0,/gladstone/finkbeiner/steve/work/data/npsad_da...,PD110_Syn1_TCx_37888x_17408y_image.png
4,/gladstone/finkbeiner/steve/work/data/npsad_da...,Pre,Pre,0.0,0.0,0.900154,1.0,/gladstone/finkbeiner/steve/work/data/npsad_da...,PD110_Syn1_TCx_38912x_3072y_image.png


In [20]:
output_df.groupby(["image_folder"])["dice/f1_score"].count()

image_folder
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/04_028_Syn1_CG_200x.svs      3
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/14_133_FCx_aSyn_x200.svs     3
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/15_134_FCx_aSyn_x200.svs     2
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/16_044_PCx_aSyn_x200.svs    13
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/PD110_Syn1_TCx.svs          18
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/PD125_Syn1_PCx.svs           9
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/PD258_Syn1_PCx.svs           2
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/PD268_Syn1_CG.svs           28
/gladstone/finkbeiner/steve

In [21]:
output_df[~output_df["dice/f1_score"].isna()].groupby(["image_folder"])["matched_label"].sum()

image_folder
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/04_028_Syn1_CG_200x.svs      3.0
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/14_133_FCx_aSyn_x200.svs     2.0
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/15_134_FCx_aSyn_x200.svs     1.0
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/16_044_PCx_aSyn_x200.svs     6.0
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/PD110_Syn1_TCx.svs          17.0
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/PD125_Syn1_PCx.svs           7.0
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/PD258_Syn1_PCx.svs           2.0
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/PD268_Syn1_CG.svs           17.0
/gladstone/

In [22]:
output_df.groupby(["image_folder"])["dice/f1_score"].mean()

image_folder
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/04_028_Syn1_CG_200x.svs     0.921334
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/14_133_FCx_aSyn_x200.svs    0.917273
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/15_134_FCx_aSyn_x200.svs    0.747618
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/16_044_PCx_aSyn_x200.svs    0.831902
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/PD110_Syn1_TCx.svs          0.890363
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/PD125_Syn1_PCx.svs          0.872158
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/PD258_Syn1_PCx.svs          0.906148
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/PD268_Syn1_CG.sv

In [23]:
ground_truth_path = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth"
#model_output_path = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Results/woven-wood-253_mrcnn_model_24.pth_3uctrn1o10.pth"
model_output_path = "/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/wgm_model_crops_pred_LBs/woven-wood-253_mrcnn_model_24.pth_3uctrn1o10.pth/"

In [24]:
def rename_image(img_crop_name):
    #x = img_crop_name.split("_")[-2]
    x = img_crop_name.split("_")[-3]
    #y = img_crop_name.split("_")[-1]
    y = img_crop_name.split("_")[-1].split(".")[0]
    #return "_".join(img_crop_name.split("_")[:-2]) + "_"+ x +"x"+"_"+y+"y"+"_image.png"
    return "_".join(img_crop_name.split("_")[:-4]) + "_"+ x +"x"+"_"+y+"y"+"_image.png"

In [25]:
file_dir = glob.glob(os.path.join(ground_truth_path,"*"))
all_csv_df = pd.DataFrame()
for file in file_dir:
    csv_path = pd.read_csv(os.path.join(model_output_path, file.split("/")[-1], file.split("/")[-1].replace("svs","csv")))
    if len(all_csv_df)==0:
        all_csv_df = csv_path
    else:
        all_csv_df = pd.concat([all_csv_df, csv_path],ignore_index=True)

In [26]:
all_csv_df.head()

,Unnamed: 0,image_name,label,confidence,brown_pixels,TRUE,Pre,FALSE,centroid,eccentricity,area,equivalent_diameter,Incorrect LB segmentations,Missing Segmentation,Bad WGM Segmentation,Incorrect Class
0,0,PD110_Syn1_TCx_x_39680_y_19968.png,Pre,0.738943,0,0,1,0,"(48.53109243697479, 28.150420168067228)",0.948132,1190,38.924993,1.0,NaN,NaN,NaN
1,0,PD110_Syn1_TCx_x_20224_y_16896.png,Pre,0.878069,0,0,1,0,"(43.41299019607843, 39.729166666666664)",0.780843,816,32.232956,1.0,NaN,NaN,NaN
2,1,PD110_Syn1_TCx_x_20224_y_16896.png,TRUE,0.669081,0,1,0,0,"(37.063079777365495, 39.183673469387756)",0.845773,539,26.196872,1.0,NaN,NaN,NaN
3,0,PD110_Syn1_TCx_x_7936_y_27136.png,Pre,0.979774,0,0,1,0,"(33.341818181818184, 34.82181818181818)",0.565106,275,18.712052,NaN,NaN,NaN,NaN
4,1,PD110_Syn1_TCx_x_7936_y_27136.png,TRUE,0.857392,0,1,0,0,"(38.99006622516556, 37.211920529801326)",0.354511,604,27.731511,NaN,NaN,NaN,NaN


In [27]:
all_csv_df["image_name"] = all_csv_df["image_name"].apply(lambda l:rename_image(l))

In [28]:
all_csv_df

,Unnamed: 0,image_name,label,confidence,brown_pixels,TRUE,Pre,FALSE,centroid,eccentricity,area,equivalent_diameter,Incorrect LB segmentations,Missing Segmentation,Bad WGM Segmentation,Incorrect Class
0,0,PD110_Syn1_TCx_39680x_19968y_image.png,Pre,0.738943,0,0,1,0,"(48.53109243697479, 28.150420168067228)",0.948132,1190,38.924993,1.0,NaN,NaN,NaN
1,0,PD110_Syn1_TCx_20224x_16896y_image.png,Pre,0.878069,0,0,1,0,"(43.41299019607843, 39.729166666666664)",0.780843,816,32.232956,1.0,NaN,NaN,NaN
2,1,PD110_Syn1_TCx_20224x_16896y_image.png,TRUE,0.669081,0,1,0,0,"(37.063079777365495, 39.183673469387756)",0.845773,539,26.196872,1.0,NaN,NaN,NaN
3,0,PD110_Syn1_TCx_7936x_27136y_image.png,Pre,0.979774,0,0,1,0,"(33.341818181818184, 34.82181818181818)",0.565106,275,18.712052,NaN,NaN,NaN,NaN
4,1,PD110_Syn1_TCx_7936x_27136y_image.png,TRUE,0.857392,0,1,0,0,"(38.99006622516556, 37.211920529801326)",0.354511,604,27.731511,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
365,1,16_044_PCx_aSyn_x200_28160x_8192y_image.png,Pre,0.843633,0,0,1,0,"(29.88607594936709, 31.911392405063292)",0.479144,79,10.029253,NaN,NaN,NaN,NaN
366,0,16_044_PCx_aSyn_x200_39424x_21504y_image.png,Pre,0.971503,0,0,1,0,"(29.89, 29.15)",0.559211,100,11.283792,NaN,NaN,NaN,NaN
367,0,16_044_PCx_aSyn_x200_10752x_24576y_image.png,Pre,0.977576,0,0,1,0,"(36.894444444444446, 36.43888888888889)",0.736686,360,21.409489,NaN,NaN,NaN,NaN
368,0,16_044_PCx_aSyn_x200_16896x_8192y_image.png,Pre,0.674289,0,0,1,0,"(31.21276595744681, 34.98936170212766)",0.88034,94,10.940042,NaN,NaN,NaN,NaN


In [29]:
all_csv_df["present_in_output"] = 1

In [30]:
image_name_df =  all_csv_df[["image_name","present_in_output"]].drop_duplicates()

In [31]:
output_df1 = pd.merge(output_df,image_name_df, on="image_name",how="left")

In [32]:
output_df

,img_crop,target_label,pred_label,target_mask_index,pred_mask_index,dice/f1_score,matched_label,image_folder,image_name
0,/gladstone/finkbeiner/steve/work/data/npsad_da...,True,True,0.0,0.0,0.864103,1.0,/gladstone/finkbeiner/steve/work/data/npsad_da...,PD110_Syn1_TCx_10240x_34816y_image.png
1,/gladstone/finkbeiner/steve/work/data/npsad_da...,Pre,Pre,0.0,0.0,0.793951,1.0,/gladstone/finkbeiner/steve/work/data/npsad_da...,PD110_Syn1_TCx_18432x_15360y_image.png
2,/gladstone/finkbeiner/steve/work/data/npsad_da...,True,True,0.0,0.0,0.946614,1.0,/gladstone/finkbeiner/steve/work/data/npsad_da...,PD110_Syn1_TCx_32768x_4096y_image.png
3,/gladstone/finkbeiner/steve/work/data/npsad_da...,True,True,0.0,0.0,0.937841,1.0,/gladstone/finkbeiner/steve/work/data/npsad_da...,PD110_Syn1_TCx_37888x_17408y_image.png
4,/gladstone/finkbeiner/steve/work/data/npsad_da...,Pre,Pre,0.0,0.0,0.900154,1.0,/gladstone/finkbeiner/steve/work/data/npsad_da...,PD110_Syn1_TCx_38912x_3072y_image.png
...,...,...,...,...,...,...,...,...,...
94,/gladstone/finkbeiner/steve/work/data/npsad_da...,True,Pre,0.0,0.0,0.906716,0.0,/gladstone/finkbeiner/steve/work/data/npsad_da...,16_044_PCx_aSyn_x200_38912x_15360y_image.png
95,/gladstone/finkbeiner/steve/work/data/npsad_da...,None,Pre,0.0,0.0,NaN,0.0,/gladstone/finkbeiner/steve/work/data/npsad_da...,16_044_PCx_aSyn_x200_39936x_19456y_image.png
96,/gladstone/finkbeiner/steve/work/data/npsad_da...,None,Pre,0.0,1.0,0.797297,0.0,/gladstone/finkbeiner/steve/work/data/npsad_da...,16_044_PCx_aSyn_x200_39936x_19456y_image.png
97,/gladstone/finkbeiner/steve/work/data/npsad_da...,Pre,Pre,0.0,0.0,NaN,1.0,/gladstone/finkbeiner/steve/work/data/npsad_da...,16_044_PCx_aSyn_x200_40960x_23552y_image.png


In [33]:
output_df1.to_csv("/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/Matched_objects_gt_pred.csv")

In [38]:
output_df1[output_df1["present_in_output"]==1]

,img_crop,target_label,pred_label,target_mask_index,pred_mask_index,dice/f1_score,matched_label,image_folder,image_name,present_in_output
34,/gladstone/finkbeiner/steve/work/data/npsad_da...,Pre,Pre,0.0,0.0,0.75162,1.0,/gladstone/finkbeiner/steve/work/data/npsad_da...,15_134_FCx_aSyn_x200_48128x_30720y_image.png,1.0


In [39]:
output_df1.groupby(["image_folder"])["present_in_output"].sum()

image_folder
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/04_028_Syn1_CG_200x.svs     0.0
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/14_133_FCx_aSyn_x200.svs    0.0
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/15_134_FCx_aSyn_x200.svs    1.0
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/16_044_PCx_aSyn_x200.svs    0.0
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/PD110_Syn1_TCx.svs          0.0
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/PD125_Syn1_PCx.svs          0.0
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/PD258_Syn1_PCx.svs          0.0
/gladstone/finkbeiner/steve/work/data/npsad_data/monika/LBD/Testing_results/ground_truth/PD268_Syn1_CG.svs           0.0
/gladstone/finkbein